Imports

In [3]:
import feedparser
import pandas as pd
from datetime import datetime
import time
import sys
import requests
from bs4 import BeautifulSoup

Fraser Data Collection

In [9]:
def scrape_fraser_timeline(url):
    """Scrapes date, title, description, and link info from the timeline."""
    try:
        # 1. Download the HTML content
        response = requests.get(url)
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)

        # 2. Parse the HTML
        soup = BeautifulSoup(response.content, 'html.parser')

        # 3. Find the main container element
        # Your target container is class="timeline-events clusterize-scroll" and id="list-container"
        timeline_container = soup.find('div', id='list-container')
        
        if not timeline_container:
            print("Error: Main timeline container not found.")
            return []

        # 4. Find all individual event rows
        # The articles are inside <div class="row event-row active">
        event_rows = timeline_container.find_all('div', class_='event-row')

        data = []
        for row in event_rows:
            # 5. Extract data points using the specific classes
            
            # Date and Source/Title: <h2 class="list-item">
            header_element = row.find('h2', class_='list-item')
            header_text = header_element.text.strip() if header_element else 'N/A'
            
            # Description/Summary: <p class="list-item">
            summary_element = row.find('p', class_='list-item')
            summary = summary_element.text.strip() if summary_element else 'N/A'
            
            # Associated Link: <ul><li><a href="..." class="list-item">
            link_element = row.find('a', class_='list-item')
            
            link_title = link_element.text.strip() if link_element else 'N/A'
            link_url = link_element['href'] if link_element else 'N/A'
            
            # Prepend the base URL if the link is relative
            if link_url != 'N/A' and link_url.startswith('/'):
                link_url = 'https://fraser.stlouisfed.org' + link_url
            
            # Split the header into Date and Source
            if '|' in header_text:
                date, source = [part.strip() for part in header_text.split('|', 1)]
            else:
                date = header_text
                source = 'N/A'

            data.append({
                'Date': date,
                'Source': source,
                'Summary': summary,
                'Associated Link Title': link_title,
                'Associated Link URL': link_url
            })

        return data

    except requests.exceptions.RequestException as e:
        print(f"An error occurred during the request: {e}")
        return []
    except Exception as e:
        print(f"An unexpected error occurred: {e}")
        return []
    
# Get Timeline Data from Fraser
# --- Execution ---
FINANCIAL_CRISIS_URL = "https://fraser.stlouisfed.org/timeline/financial-crisis"
fraser_financial_crisis_timeline_data = scrape_fraser_timeline(FINANCIAL_CRISIS_URL)
fraser_financial_crisis_timeline_df = pd.DataFrame(fraser_financial_crisis_timeline_data)
URL_COVID = "https://fraser.stlouisfed.org/timeline/covid-19-pandemic"
fraser_covid_timeline_data = scrape_fraser_timeline(URL_COVID)
fraser_covid_timeline_df = pd.DataFrame(fraser_covid_timeline_data)

# Save DataFrames to CSV files7
fraser_financial_crisis_timeline_df.to_csv('Data/fraser_financial_crisis_timeline.csv', index=False)    
fraser_covid_timeline_df.to_csv('Data/fraser_covid_timeline.csv', index=False)